# Building a RAG Knowledge Base with LangChain, ChromaDB/FAISS & the Gemini API

This notebook is a self-contained **Google Colab** version of the RAG (Retrieval-Augmented
Generation) assignment, using **Google's Gemini API** for both embeddings and the final
answer generation.

**Steps covered:**
1. Find the LangChain lib for converting text to embeddings (`GoogleGenerativeAIEmbeddings`)
2. Define our data source (sample `.txt` files, or your own uploaded files)
3. Create a Chroma (or FAISS) vector store instance
4. Convert text into embedded format and store it
5. Retrieve relevant chunks for a user query, then ask **Gemini** to generate a grounded answer

> **You will need a free Gemini API key** from https://aistudio.google.com/app/apikey — keep it secret, never commit it to a public repo.


## 0. Setup: install libraries
Run this first. It installs LangChain, the Gemini integration, and both vector store backends.

In [ ]:
!pip install -q -U \
    langchain \
    langchain-community \
    langchain-core \
    langchain-text-splitters \
    langchain-google-genai \
    langchain-chroma \
    chromadb \
    faiss-cpu \
    pypdf


### Provide your Gemini API key

Two options in Colab:
- **Recommended:** click the 🔑 icon in the left sidebar → *Secrets* → add a secret named `GOOGLE_API_KEY` with your key, then toggle "Notebook access" on. The cell below will pick it up automatically.
- **Quick & dirty:** just run the cell and paste your key when prompted (it uses `getpass`, so it won't be shown or saved in the notebook).


In [ ]:
import os

api_key = None

# Try Colab's secret manager first (Secrets tab, key icon in left sidebar)
try:
    from google.colab import userdata
    api_key = userdata.get("GOOGLE_API_KEY")
except Exception:
    pass

# Fall back to manual entry
if not api_key:
    from getpass import getpass
    api_key = getpass("Enter your Gemini API key (from https://aistudio.google.com/app/apikey): ")

os.environ["GOOGLE_API_KEY"] = api_key
print("Gemini API key is set." if os.environ.get("GOOGLE_API_KEY") else "No key set!")


## Step 1: Find the LangChain lib for converting text to embeddings

LangChain wraps every embedding provider behind the same `Embeddings` interface
(`embed_documents`, `embed_query`). For Gemini, that wrapper is
`GoogleGenerativeAIEmbeddings`, from the `langchain-google-genai` package, which calls
Google's `embedding-001` (or `text-embedding-004`) model under the hood.


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embedding_model = GoogleGenerativeAIEmbeddings(
    model="models/text-embedding-004",  # Gemini's embedding model
    google_api_key=os.environ["GOOGLE_API_KEY"],
)

# quick smoke test
test_vector = embedding_model.embed_query("What is retrieval-augmented generation?")
print("Embedding length:", len(test_vector))
print("First 5 values:", test_vector[:5])


## Step 2: Define our data source

You have two options below — use either one (or both):

**Option A — sample knowledge base** (runs immediately, no files needed): five short `.txt`
files about RAG/LangChain/vector databases get created for you.

**Option B — upload your own files** (`.txt` or `.pdf`) using Colab's file upload widget.


In [ ]:
# Option A: create a small sample knowledge base of .txt files
import os

KB_DIR = "knowledge_base"
os.makedirs(KB_DIR, exist_ok=True)

sample_docs = {
"01_what_is_rag.txt": '''Retrieval-Augmented Generation (RAG)

Retrieval-Augmented Generation, or RAG, is a technique that combines a retrieval system
with a large language model (LLM). Instead of relying only on knowledge baked into the
model during training, a RAG system first searches an external knowledge base for
relevant text, then passes that text to the LLM as context so it can generate a
grounded, up-to-date answer.

RAG reduces hallucination, lets a model answer questions about private or recent
documents, and avoids retraining the model whenever the underlying data changes. A
typical RAG pipeline has two phases: an indexing phase (split documents into chunks,
embed them, store in a vector database) and a query phase (embed the user's question,
retrieve the closest chunks, feed them to the LLM with the question).''',

"02_langchain_overview.txt": '''LangChain Overview

LangChain is a framework for building LLM-powered applications. For a RAG use case, the
most relevant building blocks are: document loaders (TextLoader, PyPDFLoader), text
splitters (RecursiveCharacterTextSplitter), embedding classes (GoogleGenerativeAIEmbeddings,
OpenAIEmbeddings, HuggingFaceEmbeddings), vector store integrations (Chroma, FAISS), and
retrievers, which expose a simple interface a chain or agent can call to fetch relevant
context.''',

"03_vector_databases.txt": '''Vector Databases: FAISS and Chroma

A vector database stores embeddings (arrays of floating point numbers) alongside the
original text and metadata, and supports similarity search - finding the stored vectors
closest to a query vector, usually via cosine similarity.

FAISS (Facebook AI Similarity Search) is a library from Meta for efficient similarity
search over large collections of dense vectors; it is lightweight and great for local
prototypes. Chroma (ChromaDB) is an open-source embedding database built for LLM apps,
with built-in persistence and metadata filtering, and integrates directly with LangChain
via langchain-chroma.''',

"04_embeddings.txt": '''Text Embeddings

An embedding is a numeric vector representation of text such that semantically similar
texts end up close together in vector space. Embedding models are trained so that
semantic similarity is reflected as geometric closeness (e.g. cosine similarity).

Gemini's text-embedding-004 model, OpenAI's text-embedding-3-small, and local
sentence-transformers models are all common choices. Once documents are embedded and
stored, a user query is embedded with the SAME model, and the vector database returns
the stored chunks whose embeddings are closest to the query embedding.''',

"05_agents.txt": '''AI Agents

An AI agent uses a large language model as a "reasoning engine" to decide which actions
to take, in what order, to accomplish a goal. Agents have access to tools (a web search
tool, a calculator, a retriever) and can call them, observe results, and decide the next
step.

RAG is often exposed to an agent as a single tool: "search the knowledge base." The
agent decides when a question needs a lookup versus when it can answer directly, calls
the retriever tool, reads the returned chunks, and composes its final answer.''',
}

for filename, content in sample_docs.items():
    with open(os.path.join(KB_DIR, filename), "w", encoding="utf-8") as f:
        f.write(content)

print(f"Created {len(sample_docs)} sample files in '{KB_DIR}/'")


In [ ]:
# Option B (optional): upload your own .txt / .pdf files into the same knowledge_base/ folder
# Uncomment and run this cell in Colab to add your own documents.

# from google.colab import files
# uploaded = files.upload()
# for fname in uploaded.keys():
#     dest = os.path.join(KB_DIR, fname)
#     os.rename(fname, dest)
#     print("Added:", dest)


In [ ]:
# Load every .txt (and .pdf, if you uploaded any) file in knowledge_base/, then split into chunks
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

txt_loader = DirectoryLoader(KB_DIR, glob="*.txt", loader_cls=TextLoader,
                              loader_kwargs={"encoding": "utf-8"})
documents = txt_loader.load()

# also pick up any PDFs you uploaded
pdf_loader = DirectoryLoader(KB_DIR, glob="*.pdf", loader_cls=PyPDFLoader)
documents += pdf_loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
chunks = splitter.split_documents(documents)

print(f"Loaded {len(documents)} document(s) -> split into {len(chunks)} chunks.")


## Step 3: Create a Chroma vector store instance

`Chroma` is LangChain's wrapper around ChromaDB. We give it our Gemini embedding model
and a `persist_directory` so the collection is saved to disk inside the Colab VM
(you can zip/download it, or re-run this notebook to rebuild it any time).


In [ ]:
from langchain_chroma import Chroma

CHROMA_DIR = "chroma_store"

vectorstore = Chroma(
    collection_name="course_knowledge_base",
    embedding_function=embedding_model,
    persist_directory=CHROMA_DIR,
)
print("Chroma collection ready at:", CHROMA_DIR)


## Step 4: Convert text into embedded format and store it in Chroma

`vectorstore.add_documents(chunks)` calls `embedding_model.embed_documents()` on every
chunk (using the Gemini API) and stores the resulting vectors + text + metadata in Chroma.

> This step makes real calls to the Gemini API — it may take a few seconds depending on
> how many chunks you have.


In [ ]:
vectorstore.add_documents(chunks)
print(f"Indexed {len(chunks)} chunks into Chroma.")


## Step 5: Retrieve data from the vector store based on a user query

`similarity_search(query, k)` embeds the query with the same Gemini embedding model and
returns the `k` closest chunks.


In [ ]:
def retrieve(query, k=3):
    return vectorstore.similarity_search(query, k=k)

user_query = "What is the difference between FAISS and Chroma?"
results = retrieve(user_query, k=3)

print(f"Query: {user_query}\n")
for i, doc in enumerate(results, start=1):
    source = os.path.basename(doc.metadata.get("source", "unknown"))
    print(f"--- Result {i} (source: {source}) ---")
    print(doc.page_content.strip()[:300])
    print()


## Bonus: complete the RAG loop — generate an answer with Gemini

Retrieval alone only gets you the relevant text. A full RAG pipeline stuffs that text
into a prompt and asks an LLM — here, **Gemini** via `ChatGoogleGenerativeAI` — to write
the final answer grounded in the retrieved context.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    google_api_key=os.environ["GOOGLE_API_KEY"],
    temperature=0.2,
)

rag_prompt = ChatPromptTemplate.from_template('''\
You are a helpful assistant answering questions using ONLY the context below.
If the answer isn't in the context, say you don't know.

Context:
{context}

Question: {question}

Answer:''')

def ask(question: str, k: int = 3) -> str:
    retrieved_docs = retrieve(question, k=k)
    context = "\n\n".join(
        f"[Source: {os.path.basename(d.metadata.get('source', 'unknown'))}]\n{d.page_content}"
        for d in retrieved_docs
    )
    chain = rag_prompt | llm
    response = chain.invoke({"context": context, "question": question})
    return response.content

print(ask("What is the difference between FAISS and Chroma?"))


In [ ]:
# Try a few more questions against your knowledge base
for q in [
    "What does RAG stand for and why is it useful?",
    "How does an AI agent decide to use a retrieval tool?",
    "Which LangChain class would I use to load a PDF?",
]:
    print("Q:", q)
    print("A:", ask(q))
    print("-" * 80)


## Alternative: swap in FAISS instead of Chroma

LangChain's vector stores share a common interface, so switching backends only touches
Steps 3–4. Everything else (embeddings, retrieval, generation) stays the same.


In [ ]:
from langchain_community.vectorstores import FAISS

# Steps 3 + 4 collapse into one call for FAISS:
faiss_store = FAISS.from_documents(chunks, embedding_model)
faiss_store.save_local("faiss_store")

# Step 5, same as before:
faiss_results = faiss_store.similarity_search("How does an AI agent decide to use a retrieval tool?", k=3)
for i, doc in enumerate(faiss_results, start=1):
    print(f"--- Result {i} (source: {os.path.basename(doc.metadata.get('source','unknown'))}) ---")
    print(doc.page_content.strip()[:300])
    print()


## Recap

| Step | What we did | LangChain / Gemini piece |
|---|---|---|
| 1 | Chose an embedding model | `GoogleGenerativeAIEmbeddings` (`text-embedding-004`) |
| 2 | Defined the data source | `DirectoryLoader` + `TextLoader`/`PyPDFLoader`, `RecursiveCharacterTextSplitter` |
| 3 | Created the vector store | `Chroma(...)` (or `FAISS.from_documents(...)`) |
| 4 | Embedded & stored chunks | `vectorstore.add_documents(chunks)` |
| 5 | Retrieved + generated an answer | `similarity_search()` + `ChatGoogleGenerativeAI` (`gemini-2.0-flash`) |

**Next steps you could try:**
- Replace the sample `knowledge_base/` files with your own course notes or PDFs (use the upload cell in Step 2).
- Add metadata filters (e.g. filter by source file) to `similarity_search`.
- Wrap `ask()` as a LangChain **tool** and give it to an **agent**, so the agent decides on its own when to search the knowledge base.
- Persist `chroma_store/` to Google Drive (`drive.mount`) so your index survives across Colab sessions.
